# Livrable 2 : Datasets, Implémentation des Algorithmes et Plan d'Expérience

## Projet CesiCDP — Résolution du TSPTW par Heuristiques

---

## I. Datasets — Génération des Instances

### 1.1 Source des Données

Dans notre projet, les instances sont entièrement **générées de manière aléatoire**.

Chaque ville est caractérisée par :

- Des coordonnées $(x, y)$ dans un plan cartésien
- Une fenêtre temporelle $[\text{earliest}, \text{latest}]$

Le **dépôt** (ville de départ) est fixé à l'identifiant 0 et possède une fenêtre temporelle très large $[0, 10000]$, ce qui le rend toujours accessible. Toutes les tournées commencent et se terminent à ce point.

Nous réalisons également l'hypothèse suivante :

$$\text{vitesse} = 1 \text{ unité de distance par unité de temps}$$

Donc :

$$\text{temps} = \frac{\text{distance}}{\text{vitesse}} = \frac{\text{distance}}{1} = \text{distance}$$

Cette modélisation suppose une **vitesse constante égale à 1**. Ainsi, toutes les grandeurs sont comparables entre elles, puisque la distance et le temps utilisent la même échelle.

De plus, les coordonnées des villes sont générées aléatoirement dans un carré $[0, 100] \times [0, 100]$, ce qui permet de simuler une répartition géographique.

Grâce à ces coordonnées, nous calculons les distances entre les villes en utilisant la **formule de distance euclidienne** :

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

### 1.2 Génération Aléatoire des Instances

Les instances sont générées aléatoirement à chaque exécution.

Pour chaque run :

- Un ensemble de villes est généré avec :
  - Des coordonnées aléatoires
  - Des fenêtres temporelles aléatoires $[\text{earliest}, \text{latest}]$, avec $\text{earliest} \in [0, 50]$ et $\text{latest} > \text{earliest}$
- Le nombre de villes $n$ est fixé selon l'expérience (entre 4 et 100)
- Le dépôt est toujours la ville 0
- Les autres villes sont cataloguées de 1 à $n-1$

Lors de l'évaluation d'une tournée :

- Si le véhicule **arrive avant** $\text{earliest}$, il **attend** (autorisé)
- Si l'arrivée **dépasse** $\text{latest}$, une **pénalité de 1000** est appliquée

Certaines arêtes entre villes sont **bloquées aléatoirement** (environ 2%). Cela signifie que :

- Certaines connexions ne sont pas autorisées
- Ces contraintes varient à chaque instance

Si une arête bloquée est empruntée, une **pénalité de 1000** est ajoutée au coût total.

Ainsi, le score total est :

$$\text{score} = \text{distance totale} + \text{pénalités}$$

## II. Implémentation des Algorithmes

Notre programme est structuré en plusieurs modules principaux :

| Module | Rôle |
| --- | --- |
| **Ville, Graphe** | Structures de données représentant les villes et le réseau routier |
| **generer_instance()** | Génération d'instances aléatoires (coordonnées, fenêtres temporelles, arêtes bloquées) |
| **evaluer_tournee()** | Calcul du score d'une tournée en intégrant les contraintes (fenêtres temporelles et arêtes bloquées) |
| **glouton()** | Algorithme heuristique du plus proche voisin (complexité $O(n^2)$) |
| **recuit()** | Algorithme de recuit simulé basé sur des mouvements 2-opt pour améliorer la solution |
| **verifier_solution()** | Vérification de la validité du cycle hamiltonien et du respect des contraintes |
| **Plan d'expérience** | Étude statistique des performances sur différentes tailles d'instances générées aléatoirement |
| **Visualisation & analyse** | Génération de graphiques et comparaison des résultats entre les algorithmes |

### 2.1 Algorithme Glouton (Nearest Neighbor)

L'algorithme glouton repose sur l'heuristique du **plus proche voisin**. À chaque étape, il choisit la ville non visitée la plus proche dont l'arête n'est pas bloquée.

Cet algorithme est rapide, avec une complexité en $O(n^2)$, mais il **ne garantit pas une solution optimale**.

**Pseudo-code :**

```
tour ← [dépôt]

Tant que des villes restent non visitées :
    candidats ← villes accessibles (arêtes non bloquées)
    prochain  ← ville la plus proche parmi les candidats
    tour.append(prochain)

tour.append(dépôt)   ← fermeture du cycle hamiltonien
```

Cet algorithme construit donc une solution rapidement, mais ne prend **pas en compte les fenêtres temporelles** lors de la construction du trajet.

### 2.2 Solveur Exact (Branch and Bound)

Le solveur exact implémente une méthode de type **Branch and Bound**. Il consiste à explorer les différentes solutions possibles (les permutations des villes), tout en évitant de tester celles qui sont inutiles. Sa complexité est de $O(n!)$ dans le pire cas.

Le principe est le suivant :

1. On construit progressivement un chemin (solution partielle)
2. À chaque étape, on calcule un coût partiel
3. Si ce coût est déjà plus mauvais que la meilleure solution trouvée, on arrête d'explorer cette branche (**élagage**)

Ce solveur prend en compte les contraintes du problème :

- Si une arête bloquée est utilisée → pénalité de **+1000**
- Si une fenêtre temporelle n'est pas respectée → pénalité de **+1000**

Ainsi, le solveur exact minimise :

$$\text{score} = \text{distance totale} + \text{pénalités}$$

> **Note :** En raison de la complexité élevée, le solveur est utilisé **uniquement pour des petites instances** (généralement $n \leq 12$).

### 2.3 Recuit Simulé

Le recuit simulé est une **méta-heuristique d'optimisation** qui améliore une solution initiale (ici, celle du glouton). Sa complexité est de $O(n \times k \times \text{itérations})$ avec $k$ = nombre de paliers (lié à $\alpha$).

Il utilise des transformations de type **2-opt**, qui consistent à améliorer un chemin en inversant une partie du trajet entre deux points pour réduire la distance totale.

L'acceptation d'une solution se fait selon le **critère de Metropolis** :

$$P(\text{accepter}) = \exp\left(\frac{-\Delta}{T}\right)$$

où :
- $\Delta$ représente le coût du changement (plus $\Delta$ est grand, plus la nouvelle solution est mauvaise)
- $T$ est la **température** (degré de liberté)
  - Au début, $T$ est élevé : on accepte souvent des solutions moins bonnes pour **explorer** toutes les possibilités
  - À la fin, $T$ est bas : on devient très strict et on n'accepte presque que les meilleures solutions

**Paramètres utilisés :**

| Paramètre | Valeur | Rôle |
| --- | --- | --- |
| $T_{\text{initial}}$ | 200.0 | Exploration large au début |
| $T_{\text{final}}$ | 0.1 | Arrêt de l'algorithme |
| $\alpha$ (alpha) | 0.995 | Refroidissement progressif |
| iterations | 50 | Nombre d'essais par température |

### 2.4 Gestion des Contraintes

Les contraintes sont gérées dans la fonction `evaluer_tournee()` à l'aide de **pénalités**.

**Fenêtres temporelles (Time Windows) :**

- Si le véhicule arrive **avant** $\text{earliest}$ → il **attend** (autorisé)
- Si le véhicule arrive **après** $\text{latest}$ → **pénalité de +1000**

**Arêtes bloquées :**

- Si une arête $(i, j)$ est interdite → **pénalité de +1000**

Ainsi, le **score total** est :

$$\text{score} = \text{distance totale} + \text{pénalités}$$

## III. Exemples d'Exécution

### 3.1 Instance Facile — Glouton

Paramètres :

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations par palier | 50 |

Voici les résultats obtenus pour **4 villes** et **0 arêtes bloquées** :

**Algorithme Glouton :**
- Cycle : `[0, 2, 3, 1, 0]`
- Score : **1075.59**

**Solveur Exact (Branch and Bound) :**
- Cycle optimal : `[0, 1, 2, 3, 0]`
- Score optimal : **72.62**

**Algorithme Recuit Simulé :**
- Cycle : `[0, 1, 2, 3, 0]`
- Score : **72.62**

Cette instance est considérée comme étant facile pour plusieurs raisons :

Le nombre de villes est faible.
Les fenêtres temporelles ne sont pas contraignantes.
Peu d’arêtes sont bloquées.

Le glouton construit une solution rapidement en choisissant toujours la ville la plus proche accessible.
Il obtient un score de 1075.59, ce qui est une mauvaise solution due à une pénalité de 1000.

Le solveur ainsi que le recuit simulé obtiennent une nette amélioration avec un score de 72.62. Bien évidemment, pour des instances à petites échelles le recuit simulé prend plus de temps que le solveur.

Ainsi, le glouton pose problème puisque bien que nous somme sur des instances jugées faciles il tombe naîvement dans le piège et obtient un haut score.
Le recuit simulé et le solveur apportent une petite amélioration seulement.


### 3.2 Instance Facile — Solveur Exact (Branch and Bound)

Paramètres :

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 1000.0 |
| Température finale | 0.1 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations par palier | 200 |

Voici les résultats obtenus pour **3 villes** et **0 arêtes bloquées** :

**Algorithme Glouton :**
- Cycle : `[0, 2, 1, 0]`
- Score : **97.23**

**Solveur Exact (Branch and Bound) :**
- Cycle optimal : `[0, 2, 1, 0]`
- Score optimal : **97.23**

**Algorithme Recuit Simulé :**
- Cycle : `[0, 2, 1, 0]`
- Score : **97.23**

Sur cette instance très simple (3 villes), les **trois algorithmes** trouvent exactement la même solution. Cela signifie que :

- Le problème est très simple (peu de villes, donc peu de combinaisons)
- Le glouton trouve déjà la solution optimale
- Le recuit simulé n'améliore pas car il n'y a rien à améliorer
- Le solveur exact confirme que cette solution est bien optimale

### 3.3 Instance Difficile — Recuit Simulé

Voici les résultats obtenus pour **74 villes** et **105 arêtes bloquées** :

**Algorithme Glouton :**
- Cycle : `[0, 2, 4, 28, 55, 11, 36, 18, 44, 62, 68, 14, 13, 25, ...]`
- Score : **64 896.90**

**Algorithme Recuit Simulé :**
- Cycle : `[0, 49, 48, 22, 67, 35, 51, 26, 61, 37, 24, 15, ...]`
- Score : **61 738.54**

Cette instance est considérée comme **difficile** car :

- Il y a beaucoup de villes (74), donc énormément de solutions possibles
- Un nombre accru d'arêtes bloquées et des fenêtres temporelles dures à respecter obligent à visiter les villes dans un ordre très précis

Le recuit simulé améliore la solution en utilisant la méthode 2-opt pour tester de nouveaux chemins, et trouve progressivement un chemin qui respecte mieux les fenêtres temporelles. Le score passe de **64 896.90** (glouton) à **61 738.54** (recuit).

## IV. Plan d'Expérience

Dans ce plan d'expérience, nous comparons les performances de l'algorithme glouton et du recuit simulé sur plusieurs tailles d'instances.

**Protocole expérimental :**

- Les instances sont générées de manière aléatoire à chaque exécution
- Pour chaque taille, nous réalisons **30 runs** afin d'obtenir des résultats représentatifs (moyenne, écart-type, minimum, maximum)
- Une graine aléatoire différente est utilisée à chaque run : $\text{graine} = \text{seed} \times 13 + n$
- Cela garantit la **reproductibilité** tout en assurant des instances variées

### 4.1 Expérience n°1 — Variation du Facteur de Refroidissement $\alpha$

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Nombre d'itérations par palier | 50 |
| Taille | $n = 30$ villes |

**Résultats :**

| Alpha | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0.95 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.7 | |
| | Recuit | 22 965.30 | 995.67 | 21 487.35 | 25 562.34 | 1 055.4 | 8.0% |
| 0.99 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.6 | |
| | Recuit | 22 100.98 | 847.07 | 20 474.38 | 23 590.62 | 5 227.3 | 11.5% |
| 0.995 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.7 | |
| | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 10 422.4 | 12.2% |

**Analyse :** On observe que le recuit simulé améliore les résultats du glouton pour toutes les valeurs de α.

Cependant, l’impact du paramètre α est très significatif sur trois facteurs :

-- la qualité de la solution
-- la stabilité (écart-type)
-- le temps de calcul

On observe que la qualité de la solution s’améliore lorsque α augmente jusqu’à 0.995, mais se dégrade légèrement pour α = 0.999.
Cela montre qu’il existe un résultat optimal pour le paramètre α, ici autour de 0.995.

En prenant $\text{Opt}(D) \approx 21 933.11$ comme référence :

$$\text{Performance glouton} = \frac{24968.99}{21933.11} \approx 1.1384 \quad (\text{+13.84\% au-dessus de l'optimum})$$

$$\text{Performance recuit } (\alpha = 0.95) = \frac{22965.30}{21933.11} \approx 1.0471 \quad (\text{+4.71\%})$$

$$\text{Performance recuit } (\alpha = 0.99) = \frac{22100.98}{21933.11} \approx 1.0077 \quad (\text{+0.77\%})$$

$$\text{Performance recuit } (\alpha = 0.995) = \frac{21933.11}{21933.11} = 1.000 \quad (\text{référence})$$

Le facteur α contrôle la vitesse de refroidissement :

Plus α est faible (0.95) :

- refroidissement rapide
- exploration limitée
- rapide mais moins optimal

Plus α est élevé (0.995) :
- refroidissement lent 
- exploration approfondie
- meilleure solution mais très long (Par exemple alpha = 0.995)

Mais si α est trop élevé (0.999), le temps devient très long, sans amélioration significative

Nous pouvons donc conclure que le recuit simulé reste toujours meilleur que le glouton, quel que soit le alpha. Cependant, le choix de α influence fortement les performances.
Un α trop faible limite l’exploration, tandis qu’un α trop élevé augmente fortement le temps de calcul sans gain significatif.
Le meilleur compromis observé est pour α = 0.995.

### 4.2 Expérience n°2 — Variation du Nombre d'Itérations

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Taille | $n = 30$ villes |

**Résultats :**

| Itérations | Algo | Moyenne | Écart-type | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- |
| 20 | Recuit | 40 683.52 | 1 796.19 | 881.9 | 4.5% |
| 30 | Recuit | 40 651.58 | 2 012.87 | 1 206.1 | 4.6% |
| 50 | Recuit | 40 250.96 | 1 739.45 | 1 114.7 | 5.5% |
| 80 | Recuit | 40 546.64 | 1 359.99 | 3 493.6 | 4.9% |

Dans cette deuxième expérience, on fait varier le nombre d’itérations de l’algorithme du recuit simulé pour voir son impact sur les résultats.

On remarque que la qualité des solutions reste assez stable. La moyenne des coûts change légèrement selon le nombre d’itérations, mais les différences ne sont pas très importantes.

**Analyse :** La qualité des solutions reste stable. En prenant $\text{Opt}(D) \approx 40 250.96$ :

$$\frac{40683.52}{40250.96} \approx 1.0099 \quad (\text{+0.99\% avec 30 itérations})$$

$$\frac{40546.64}{40250.96} \approx 1.0073 \quad (\text{+0.73\% avec 80 itérations})$$

Nous pouvons donc observer que le meilleur résultat est obtenu avec 50 itérations. Les écarts sont très faibles (moins de 1.1%). De plus augmenter les itérations ne garantit pas une amélioration.

Ainsi, le recuit simulé est peu sensible au nombre d’itérations dans cette plage, et la solution reste globalement stable.

### 4.3 Expérience n°3 — Variation de la Graine Aléatoire

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations | 50 |
| Taille | $n = 30$ villes |

**Résultats :**

| Graine | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 13 | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 9 367.1 | 12.2% |
| 52 | Recuit | 22 135.57 | 869.17 | 19 507.18 | 24 477.16 | 8 370.2 | 10.1% |
| 78 | Recuit | 21 843.29 | 884.41 | 19 479.69 | 23 552.18 | 8 373.7 | 10.2% |

Dans cette expérience, on observe que les résultats du recuit simulé restent très proches d’une graine à l’autre.

Les moyennes varient légèrement entre 21 843 et 22 135
Les écarts-types restent du même ordre de grandeur
Les temps d’exécution sont également très proches

L’amélioration par rapport au glouton reste comprise entre 10.1% et 12.2%, ce qui montre une bonne stabilité des performances

**Analyse :** En prenant $\text{Opt}(D) \approx 21 843.29$ (meilleure solution) :

$$\frac{21843.29}{21843.29} \approx 1.0000 \quad (\text{référence, graine 78})$$

$$\frac{21933.11}{21843.29} \approx 1.0041 \quad (\text{+0.41\%, graine 13})$$

$$\frac{22135.57}{21843.29} \approx 1.0134 \quad (\text{+1.34\%, graine 52})$$

Donc, cette expérience montre que la graine aléatoire n’a pas d’impact significatif sur la qualité globale des solutions.

Le recuit simulé conserve une amélioration stable par rapport au glouton, ce qui confirme sa robustesse face aux différentes instances générées.

On peut conclure que changer la graine aléatoire n’a pas beaucoup d’impact sur la qualité des solutions obtenues.

### 4.4 Expérience n°4 — Variation de la Température Initiale $T_{\text{initial}}$

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations | 50 |
| Taille | $n = 30$ villes |

**Résultats :**

| $T_{\text{initial}}$ | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 50 | Recuit | 22 167.86 | 830.70 | 20 483.95 | 24 491.27 | 605.5 | 11.2% |
| 100 | Recuit | 22 101.46 | 885.27 | 19 501.52 | 23 540.53 | 687.3 | 11.5% |
| 200 | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 814.6 | 12.2% |
| 500 | Recuit | 22 007.46 | 1 002.59 | 19 460.54 | 24 595.32 | 956.1 | 11.9% |

Dans cette expérience, nous faisons varier la température initiale T_initial dans le but d’analyser son impact sur l’exploration du recuit simulé.

On observe que le recuit simulé améliore systématiquement les résultats du glouton, quelle que soit la valeur de la température initiale.
Cependant, l’impact de T_initial reste modéré. En effet, la qualité des solutions s’améliore légèrement jusqu’à T = 200 au delà (T = 500), nous pouvons observer qu'il n'y a plus de gain significatif et le temps de calcul augmente progressivement

Ainsi lorsque le T est faible (50) l'exploration devient limitée, les solutions deviennent moins bonnes. En revanche lorsque T est optimale (200) nous possédons un bon équilibre entre 
exploration limitée ente l'exploration et la convergence des données. Enfin lorsque T est élevée l'exploration est longue et il n'ya pas de gain réel.

**Analyse :** En prenant $\text{Opt}(D) \approx 21 933.11$ :

$$\frac{22101.46}{21933.11} \approx 1.0077 \quad (\text{+0.77\% pour } T = 100)$$

$$\frac{21933.11}{21933.11} = 1 \quad (\text{référence pour } T = 200)$$

$$\frac{22007.46}{21933.11} \approx 1.0034 \quad (\text{+0.34\% pour } T = 500)$$

La performance s'améliore jusqu'à $T = 200$, puis diminue légèrement. Le **meilleur compromis** est $T_{\text{initial}} \approx 200$.

Donc, la performance du recuit s’améliore lorsque T_initial augmente jusqu’à 200. En efft le le meilleur résultat est atteint pour cette température initiale. Cela nous confirme qu'une température faible limite l’exploration
et une température trop élevée n’apporte pas de gain significatif

### 4.5 Expérience n°5 — Variation du Nombre de Villes

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations | 50 |
| Nombre de runs | 30 |

**Résultats :**

| $n$ (villes) | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 30 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.8 | |
| | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 11 008.6 | 12.2% |
| 60 | Glouton | 52 258.60 | 1 615.95 | 49 669.18 | 55 919.78 | 10.7 | |
| | Recuit | 49 941.49 | 1 227.64 | 46 696.66 | 51 866.57 | 22 159.0 | 4.4% |
| 80 | Glouton | 71 350.15 | 2 021.48 | 67 868.72 | 76 878.30 | 18.9 | |
| | Recuit | 69 375.98 | 1 337.97 | 66 875.26 | 71 899.33 | 27 319.1 | 2.8% |

On observe plusieurs phénomènes importants, le score augmente avec la taille. En effet, quand le nombre de villes augmente, le score du glouton augmente fortement, le score du recuit aussi. C’est donc tout à fait logique puisque plus il y a de villes, plus il y a de distance à parcourir.

Malheureusement, lorsque la taille est petite le recuit explore presque tout, lorsque n devient grand il possède donc un espace de recherche énorme, et ainsi une exploration limitée.


**Analyse :** Lorsque la taille augmente :

- Le score augmente (logique : plus de villes = plus de distance)
- L'amélioration du recuit **diminue progressivement** : 12.2% → 4.4% → 2.8%
- Le recuit reste toujours meilleur que le glouton, mais son avantage se réduit

Cela s'explique par l'augmentation de la complexité du problème, qui **limite la capacité d'exploration** du recuit.

## V. Interprétation des Résultats

### 5.1 Rôle du Facteur $\alpha$

On observe que le paramètre α (facteur de refroidissement) joue un rôle essentiel dans le fonctionnement du recuit simulé.

Lorsque α est élevé (proche de 1), la température diminue lentement. L’algorithme explore alors davantage de solutions et accepte plus facilement des solutions moins bonnes de manière temporaire. Cela lui permet d’éviter de rester bloqué dans un minimum local et d’augmenter ses chances de trouver une meilleure solution globale. En revanche, cette exploration plus approfondie entraîne un temps de calcul beaucoup plus important.

À l’inverse, lorsque α est plus faible, la température diminue rapidement. L’algorithme se stabilise plus vite et explore moins de solutions. Il devient donc plus rapide, mais il risque de rester bloqué dans une solution de qualité moyenne.

Ainsi, α permet de contrôler un compromis fondamental :

un α élevé → meilleure qualité de solution mais temps de calcul élevé
un α faible → exécution rapide mais solution moins optimale


Les différentes expériences montrent que :

Le nombre d’itérations a un impact limité sur : la qualité des solutions puisque celle-ci varie peu, même si le temps de calcul augmente.

La température initiale (T_initial) a un impact modéré sur des valeurs trop faible qui limite l’exploration, tandis qu’une valeur trop élevée n’apporte pas d’amélioration significative mais augmente le temps de calcul.

Le meilleur compromis observé est T_initial ≈ 200.

La graine aléatoire a très peu d’influence sur les performances globales car les résultats restent stables, ce qui montre que le recuit simulé est robuste face à la variabilité des instances.

Lorsque le nombre de villes augmente, le coût des solutions augmente fortement ce qui est logique dans notre cas.
L’amélioration apportée par le recuit diminue progressivement :
12% pour n = 30
4% pour n = 60
3% pour n = 80

Ainsi, le recuit simulé reste meilleur que le glouton, mais son avantage diminue lorsque la taille du problème augmente.

Les performances dépendent également des instances générées aléatoirement puisque certaines instances sont simples (peu de contraintes) et d’autres sont plus complexes (routes bloquées, contraintes temporelles).

Le glouton est donc très sensible à ces variations car il fait des choix locaux sans remise en question.

Le recuit simulé, en revanche, est plus robuste car il explore plusieurs solutions et il peut corriger une mauvaise solution initiale.

Cela se traduit par un écart-type plus faible aissi que des résultats plus stables.

Nous avons réalisé 30 runs pour chaque expérience dans le but d’obtenir des résultats fiables.

En effet, comme les instances sont générées aléatoirement, un seul test ne serait pas représentatif. Il pourrait correspondre à un cas particulier (facile ou difficile).

En répétant les expériences, on obtient :

une moyenne correpondant à la performance globale de notre algorithme
un écart-type correpondant à la stabilité de notre algorithme
un minimum et un maximum correpondant à la variabilité de notre algorithme

Cela permet de réduire l’impact du hasard et d’obtenir une analyse plus rigoureuse.

## VI. Étude des Statistiques

### 6.1 Synthèse par Expérience

**Expérience n°1 ($\alpha$) :**

Ici, la taille est fixée (n = 30), ce qui permet d’isoler l’effet du paramètre α.

On observe que :

lorsque α augmente, le score moyen diminue légèrement car les solutions sont meilleures,
l’écart-type diminue globalement puisque les résultats deviennent plus stables
le temps de calcul augmente fortement

Ainsi α est un paramètre clé du recuit simulé. Le meilleur compromis que nous pouvons obtenir est autour de α = 0.995.

**Expérience n°2 (itérations) :**

Dans cette expérience, nous avons fait varier le nombre d’itérations par palier.

Nous observons que :

la moyenne des résultats varie très peu,
les écarts de performance restent très faibles (moins de 1%),
le temps de calcul augmente fortement avec le nombre d’itérations.

Le paramètre iterations a un impact limité sur la qualité de nos solutions dans cette plage. Il permet d’affiner légèrement, mais n’est pas déterminant. Pour trouver un bon compromis nous choisissons 50 itérations.

**Expérience n°3 (graine) :**

La graine aléatoire permet de générer différentes instances du problème.

On observe que :

les résultats du recuit restent globalement proches d’une graine à l’autre (moyenne comprise entre 21 843 et 22 135),
l’amélioration reste stable (≈ 10.1% à 12.2%),
les variations de moyenne et d’écart-type sont faibles.

Ainsi, la graine a peu d’impact sur la qualité globale des résultats. Le recuit simulé fournit des performances stables et qui sont également reproductibles, ce qui nous est essentiel.

**Expérience n°4 ($T_{\text{initial}}$) :**

On observe que lorsque le nombre de villes augmente, le score moyen des deux algorithmes augmente fortement. Cela est logique : plus il y a de villes à visiter, plus la distance totale de la tournée est élevée.

Cependant, l’élément le plus intéressant est l’évolution de l’écart-type.

On remarque que :

l’écart-type du glouton est plus élevé, surtout lorsque la taille augmente,
le recuit simulé présente un écart-type plus faible.

Cela signifie que le glouton est très sensible à l’instance générée (positions des villes, routes bloquées, contraintes). Certaines configurations peuvent fortement dégrader sa performance.

À l’inverse, le recuit simulé est plus robuste : il est capable de corriger une mauvaise solution initiale en explorant d’autres possibilités.

Ainsi la taille de l’instance influence directement la qualité des solutions et la capacité du recuit à converger vers une bonne solution.

**Expérience n°5 (taille) :**

Dans cette expérience, on fait varier la taille de l’instance (n = 30, 60, 80).

On observe que :

les scores augmentent avec la taille (comportement attendu),
le temps de calcul du recuit augmente fortement,
l’amélioration du recuit diminue lorsque n augmente :
≈ 12.2% pour n = 30
≈ 4.4% pour n = 60
≈ 2.8% pour n = 80

Le recuit simulé reste meilleur que le glouton pour toutes les tailles, mais son avantage diminue avec la taille du problème

## VII. Conclusions et Recommandations

Le choix des paramètres du recuit simulé dépend directement de l'objectif : **qualité de la solution** ou **rapidité d'exécution**.

Pour **privilégier la qualité**, nous recommandons :

| Paramètre | Valeur recommandée |
| --- | --- |
| $\alpha$ | ≈ 0.995 (élevé) |
| $T_{\text{initial}}$ | ≈ 200.0 |
| iterations | ≈ 50 |

Ces paramètres permettent :

- Une exploration approfondie
- D'éviter les minima locaux
- D'obtenir les meilleures performances

**L'inconvénient** est le temps de calcul élevé.

> **Conclusion :** Le recuit simulé est un algorithme flexible dont les performances dépendent fortement de ses paramètres. Il n'existe pas de "meilleur réglage universel" — le choix dépend du contexte et des contraintes de l'application.

## VIII. Code Source — Implémentation Complète

Le code suivant implémente l'ensemble des modules décrits dans ce livrable.

In [ ]:
import csv
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. MODÈLE — Structures de données
# ============================================================

class Ville:
    def __init__(self, id, x, y, earliest, latest):
        self.id = id
        self.x = x
        self.y = y
        self.earliest = earliest
        self.latest = latest

class Graphe:
    def __init__(self, villes):
        self.villes = villes
        self.n = len(villes)
        self.bloquees = set()

    def distance(self, i, j):
        v1, v2 = self.villes[i], self.villes[j]
        return math.hypot(v1.x - v2.x, v1.y - v2.y)

    def accessible(self, i, j):
        return (i, j) not in self.bloquees

# ============================================================
# 2. GÉNÉRATION D'INSTANCES ALÉATOIRES
# ============================================================

def generer_instance(n, ratio_bloquees=0.02, graine=None):
    """
    Génère une instance aléatoire complète avec n villes.

    Paramètres :
        n              : nombre de villes (dépôt inclus)
        ratio_bloquees : fraction des arêtes bloquées
        graine         : pour la reproductibilité

    Choix de modélisation :
        - Coordonnées dans [0, 100] (grille carrée)
        - Dépôt au centre (50, 50)
        - Fenêtres temporelles cohérentes : ouverture aléatoire
          dans [0, 50], fermeture dans [ouverture+20, 150]
        - Arêtes bloquées sur des paires non-dépôt uniquement
          pour garantir l'existence d'une solution
    """
    if graine is not None:
        random.seed(graine)
        np.random.seed(graine)

    villes = []
    # Dépôt au centre, toujours ouvert
    villes.append(Ville(0, 50.0, 50.0, 0, 10000))

    for i in range(1, n):
        x = round(random.uniform(0, 100), 2)
        y = round(random.uniform(0, 100), 2)
        earliest = round(random.uniform(0, 50), 1)
        latest = round(earliest + random.uniform(20, 100), 1)
        villes.append(Ville(i, x, y, earliest, latest))

    g = Graphe(villes)

    # Arêtes bloquées (uniquement entre villes non-dépôt)
    paires = [(i, j) for i in range(1, n)
                     for j in range(1, n) if i != j]
    if paires:
        nb_bloquees = int(len(paires) * ratio_bloquees)
        for (i, j) in random.sample(paires, nb_bloquees):
            g.bloquees.add((i, j))

    return g

# ============================================================
# 3. ÉVALUATION
# ============================================================

def evaluer_tournee(tour, g):
    """
    Évalue un cycle hamiltonien.
    Retourne le coût total + pénalités contraintes.
    """
    temps = 0
    cout = 0
    penalite = 0
    for i in range(len(tour) - 1):
        a, b = tour[i], tour[i + 1]
        # Contrainte 2 : arête bloquée → pénalité 1000
        if not g.accessible(a, b):
            penalite += 1000
        d = g.distance(a, b)
        cout += d
        temps += d
        # Contrainte 1 : fenêtre temporelle
        if temps < g.villes[b].earliest:
            temps = g.villes[b].earliest  # attente autorisée
        if temps > g.villes[b].latest:
            penalite += 1000  # violation → pénalité 1000
    return cout + penalite

# ============================================================
# 4. GLOUTON — Cycle Hamiltonien (Nearest Neighbor)
# ============================================================

def glouton(g):
    non_visite = list(range(1, g.n))
    tour = [0]

    while non_visite:
        courant = tour[-1]
        accessibles = [x for x in non_visite if g.accessible(courant, x)]
        candidats = accessibles if accessibles else non_visite
        prochain = min(candidats, key=lambda x: g.distance(courant, x))
        tour.append(prochain)
        non_visite.remove(prochain)

    tour.append(0)
    return tour

# ============================================================
# 5. SOLVEUR EXACT — Branch and Bound
# ============================================================

def solveur_exact(g, PENALITE=1000):
    """
    Solveur exact par Branch and Bound (via PuLP/CBC).
 
    Contraintes MOLLES (identique au B&B maison) :
        - Violer une arête bloquée  → pénalité 1000 dans l'objectif
        - Arriver après 'latest'    → pénalité 1000 dans l'objectif
        - Arriver avant 'earliest'  → attente autorisée (gratuite)
 
    Retourne : (tour, score, status) où :
        status = 'Optimal'   → optimum prouvé
        status = 'Timeout'   → meilleure solution trouvée, pas prouvée optimale
        status = 'Infeasible' → aucune solution
    """
    import pulp
 
    n = g.n
    villes = list(range(n))
 
    # ── Variables de décision ──
    x = pulp.LpVariable.dicts("x", (villes, villes), 0, 1, pulp.LpBinary)
    u = pulp.LpVariable.dicts("u", villes, 0, n - 1, cat='Integer')
    t = pulp.LpVariable.dicts("t", villes, lowBound=0, cat='Continuous')
 
    # Variables de violation (pénalisées)
    v_bloc = {(i, j): pulp.LpVariable(f"v_bloc_{i}_{j}", 0, 1, pulp.LpBinary)
              for (i, j) in g.bloquees}
    v_tw   = pulp.LpVariable.dicts("v_tw", villes, 0, 1, pulp.LpBinary)
    retard = pulp.LpVariable.dicts("retard", villes, lowBound=0, cat='Continuous')
 
    # ── Modèle ──
    model = pulp.LpProblem("TSPTW_BranchAndBound", pulp.LpMinimize)
 
    # Objectif : distance + pénalités
    model += (
        pulp.lpSum(g.distance(i, j) * x[i][j]
                   for i in villes for j in villes if i != j)
        + PENALITE * pulp.lpSum(v_bloc[(i, j)] for (i, j) in g.bloquees)
        + PENALITE * pulp.lpSum(v_tw[i] for i in villes)
    )
 
    # Flot
    for i in villes:
        model += pulp.lpSum(x[i][j] for j in villes if j != i) == 1
    for j in villes:
        model += pulp.lpSum(x[i][j] for i in villes if i != j) == 1
    for i in villes:
        model += x[i][i] == 0
 
    # MTZ : élimination des sous-tours
    model += u[0] == 0
    for i in villes:
        for j in villes:
            if i != j and j != 0:
                model += u[i] - u[j] + n * x[i][j] <= n - 1
 
    # Arêtes bloquées
    for (i, j) in g.bloquees:
        model += v_bloc[(i, j)] >= x[i][j]
 
    # Big-M pour les temps
    M = (sum(g.distance(i, j) for i in villes for j in villes if i != j)
         + max(v.latest for v in g.villes) + PENALITE + 1)
 
    model += t[0] == 0
    for i in villes:
        if i != 0:
            model += t[i] >= g.villes[i].earliest
 
    for i in villes:
        for j in villes:
            if i != j and j != 0:
                model += t[j] >= t[i] + g.distance(i, j) - M * (1 - x[i][j])
 
    # Retard / violation TW
    # (la ligne v_tw >= retard/M est RETIRÉE : redondante et mal conditionnée)
    for i in villes:
        if i != 0:
            model += retard[i] >= (t[i] - g.villes[i].latest)
            model += retard[i] <= M * v_tw[i]
            model += retard[i] >= 0
            model += v_tw[i] >= retard[i] / M   # ← IMPORTANT
 
    # ── Résolution ──
    solver = pulp.PULP_CBC_CMD(msg=0)
    model.solve(solver)
 
    statut_brut = pulp.LpStatus[model.status]
 
  # ── Récupération valeur objectif ──
    obj_val = pulp.value(model.objective)

# ── Détection du timeout ──
    timeout = False
    try:
        best_bound = model.solverModel.best_bound
        obj_val    = pulp.value(model.objective)
        if obj_val is not None and best_bound is not None:
            gap = abs(obj_val - best_bound) / max(abs(obj_val), 1e-6)
            timeout = gap > 1e-4
        else:
            timeout = False
    except AttributeError:
        # Fallback si l'attribut n'existe pas selon la version de PuLP
        timeout = False
 
    if obj_val is None:
        return None, float('inf'), 'Infeasible'
 
    status = 'Timeout' if timeout else 'Optimal'
 
    # ── Reconstruction du tour ──
    tour = [0]
    current = 0
    vus = {0}
    while len(tour) < n:
        trouve = False
        for j in villes:
            val = pulp.value(x[current][j])
            if (j != current and j not in vus
                    and val is not None and val > 0.5):
                tour.append(j); vus.add(j); current = j
                trouve = True; break
        if not trouve:
            return None, float('inf'), 'Infeasible'
    tour.append(0)
 
    score = evaluer_tournee(tour, g)
    return tour, score, status


# ============================================================
# 7. VÉRIFICATION
# ============================================================

def verifier_solution(tour, g):
    """Vérifie qu'un cycle hamiltonien visite toutes les villes exactement une fois."""
    villes_visitees = tour[1:-1]
    uniques = set(villes_visitees)

    print(f"  Nombre de visites : {len(villes_visitees)}")
    print(f"  Nombre unique     : {len(uniques)}")
    print(f"  Nombre attendu    : {g.n - 1}")

    if len(uniques) != g.n - 1 or len(villes_visitees) != g.n - 1:
        print("  ERREUR : villes manquantes ou doublons")
    else:
        print(" Le cycle hamiltonien est valide")

    violations = [(tour[k], tour[k+1]) for k in range(len(tour)-1)
                  if (tour[k], tour[k+1]) in g.bloquees]
    if violations:
        print(f"  Arêtes bloquées empruntées (pénalisées) : {violations}")
    else:
        print("  Aucune arête bloquée empruntée")

In [ ]:
# ============================================================
# 8. TESTS UNITAIRES DES CONTRAINTES
# ============================================================

print("=" * 55)
print("  TESTS UNITAIRES DES CONTRAINTES")
print("=" * 55)

# Test Contrainte 1 : Fenêtres Temporelles
print("\n----- TEST CONTRAINTE 1 : Fenêtres Temporelles -----")
villes_tw = [
    Ville(0, 0, 0, 0, 1000),
    Ville(1, 10, 0, 0, 1000),
    Ville(2, 50, 0, 0, 0.001),  # fenêtre IMPOSSIBLE
]
g_tw = Graphe(villes_tw)
score_sans = evaluer_tournee([0, 1, 0], g_tw)
score_avec = evaluer_tournee([0, 1, 2, 0], g_tw)
print(f"  Score sans violation TW  : {score_sans:.2f}")
print(f"  Score avec violation TW  : {score_avec:.2f}")
if score_avec > score_sans + 999:
    print("  ✅ Contrainte TW active : pénalité de 1000 bien appliquée")
else:
    print("  ❌ Contrainte TW non détectée")

# Test Contrainte 2 : Arêtes Bloquées
print("\n----- TEST CONTRAINTE 2 : Arêtes Bloquées -----")
villes_bl = [
    Ville(0, 0, 0, 0, 1000),
    Ville(1, 10, 0, 0, 1000),
    Ville(2, 20, 0, 0, 1000),
]
g_bl = Graphe(villes_bl)
tour_bl = [0, 1, 2, 0]
score_libre  = evaluer_tournee(tour_bl, g_bl)
g_bl.bloquees.add((1, 2))
score_bloque = evaluer_tournee(tour_bl, g_bl)
print(f"  Score sans arête bloquée : {score_libre:.2f}")
print(f"  Score avec arête bloquée : {score_bloque:.2f}")
if score_bloque > score_libre + 999:
    print("  ✅ Contrainte arête bloquée active : pénalité de 1000 bien appliquée")
else:
    print("  ❌ Contrainte arête bloquée non détectée")

In [ ]:
# ============================================================
# 9. RÉSOLUTION SUR INSTANCE ALÉATOIRE
# ============================================================

print("\n" + "=" * 55)
print("  RÉSOLUTION — CYCLE HAMILTONIEN")
print("=" * 55)

n_impose = 4  # Nombre de villes imposé

if n_impose is not None:
    n_instance = n_impose
    print(f"\nNombre de villes imposé : {n_instance}")
else:
    n_instance = random.randint(3, 100)
    print(f"\nNombre de villes tiré aléatoirement : {n_instance}")

g = generer_instance(n_instance)
print(f"Instance générée : {g.n} villes, {len(g.bloquees)} arêtes bloquées")

# Glouton
print("\n----- GLOUTON -----")
t1 = glouton(g)
print(f"  Cycle : {t1}")
print(f"  Score : {evaluer_tournee(t1, g):.2f}")

print("\n----- SOLVEUR EXACT — Branch and Bound -----")
SEUIL_EXACT = 30  # B&B + MTZ fiable jusqu'à ~15 villes
if g.n <= SEUIL_EXACT:
    t0 = time.perf_counter()
    tour_exact, score_exact, status = solveur_exact(g)
    duree = time.perf_counter() - t0
    if status == 'Optimal':
        print(f"  Cycle optimal : {tour_exact}")
        print(f"  Score optimal : {score_exact:.2f}  [optimum prouvé]")
    elif status == 'Timeout':
        print(f"  ⚠ Timeout atteint — solution sous-optimale")
        print(f"  Meilleure solution trouvée : {score_exact:.2f}")
        print(f"  (ne pas utiliser comme référence d'optimalité)")
        score_exact = None  # pour ne pas comparer les autres méthodes à un faux optimum
    else:
        print(f"  ⚠ Aucune solution trouvée ({status})")
        score_exact = None
    print(f"  Temps : {duree*1000:.1f} ms")
else:
    print(f"  Instance trop grande ({g.n} villes > {SEUIL_EXACT}).")
    print(f"  Formulation MTZ + Big-M non fiable au-delà.")
    score_exact = None

# Recuit Simulé
print("\n----- RECUIT SIMULÉ -----")
t2, historique = recuit(g)
print(f"  Cycle : {t2}")
print(f"  Score : {evaluer_tournee(t2, g):.2f}")
if g.n <= 12 and 'score_exact' in dir():
    score_r = evaluer_tournee(t2, g)
    gap_r = (score_r - score_exact) / score_exact * 100 if score_exact > 0 else 0
    print(f"  Performance recuit  : {score_r/score_exact:.3f} (écart : {gap_r:.1f}%)")

# Vérification
print("\n----- VÉRIFICATION -----")
verifier_solution(t2, g)

In [ ]:
# ============================================================
# 10. PLAN D'EXPÉRIENCE — Tailles variables
# ============================================================

print("\n" + "=" * 55)
print("   PLAN D'EXPÉRIENCE — Tailles variables")
print("=" * 55)

tailles = [30]
nb_runs = 30

for n in tailles:
    scores_g, scores_r, temps_g, temps_r = [], [], [], []

    for seed in range(nb_runs):
        instance = generer_instance(n, graine=seed * 13 + n)

        t0 = time.perf_counter()
        t1 = glouton(instance)
        temps_g.append(time.perf_counter() - t0)
        scores_g.append(evaluer_tournee(t1, instance))

        t0 = time.perf_counter()
        t2, _ = recuit(instance, t_initial=200.0, t_final=1.0,
                       alpha=0.995, iterations=50)
        temps_r.append(time.perf_counter() - t0)
        scores_r.append(evaluer_tournee(t2, instance))

    print(f"\nn={n} villes ({nb_runs} runs) :")
    print(f"  Glouton — moy: {np.mean(scores_g):.2f}  "
          f"écart-type: {np.std(scores_g):.2f}  "
          f"min: {np.min(scores_g):.2f}  max: {np.max(scores_g):.2f}  "
          f"temps moy: {np.mean(temps_g)*1000:.1f} ms")
    print(f"  Recuit  — moy: {np.mean(scores_r):.2f}  "
          f"écart-type: {np.std(scores_r):.2f}  "
          f"min: {np.min(scores_r):.2f}  max: {np.max(scores_r):.2f}  "
          f"temps moy: {np.mean(temps_r)*1000:.1f} ms")
    amelio = (np.mean(scores_g) - np.mean(scores_r)) / np.mean(scores_g) * 100
    print(f"  Amélioration recuit vs glouton : {amelio:.1f}%")

In [ ]:
# ============================================================
# 11. VISUALISATION — Comparaison Glouton vs Recuit Simulé
# ============================================================

moy_g, moy_r, std_g, std_r = [], [], [], []

for n in tailles:
    scores_g, scores_r = [], []
    for seed in range(nb_runs):
        instance = generer_instance(n, graine=seed * 13 + n)
        t1 = glouton(instance)
        scores_g.append(evaluer_tournee(t1, instance))
        t2, _ = recuit(instance, t_initial=200.0, t_final=1.0,
                       alpha=0.995, iterations=50)
        scores_r.append(evaluer_tournee(t2, instance))
    moy_g.append(np.mean(scores_g))
    moy_r.append(np.mean(scores_r))
    std_g.append(np.std(scores_g))
    std_r.append(np.std(scores_r))

plt.figure(figsize=(10, 5))
plt.errorbar(tailles, moy_g, yerr=std_g, marker="o", color="#2980b9",
             label="Glouton", capsize=5, lw=2)
plt.errorbar(tailles, moy_r, yerr=std_r, marker="s", color="#e74c3c",
             label="Recuit Simulé", capsize=5, lw=2)
plt.xlabel("Nombre de villes")
plt.ylabel("Score moyen")
plt.title("Comparaison Glouton vs Recuit Simulé\n(moyenne ± écart-type sur 30 runs)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("plan_experience.png", dpi=150, bbox_inches="tight")
print("\nGraphique sauvegardé : plan_experience.png")
plt.show()